## Importing Modules

In [1]:
# Local workspace setup (no clone needed when running inside this repo).
import os
print('Working directory:', os.getcwd())

Working directory: /Users/kausar/Documents/moment-to-action


In [2]:
# macOS local run: apt-get is not required here.
print('Skipping system package install on macOS.')

Skipping system package install on macOS.


In [3]:
# Install dependencies from requirements file located deeper in the repo structure.
# Adjust path relative to CWD: /Users/kausar/Documents/moment-to-action
%pip install -r movinet/violence-streaming/requirements_tflite.txt

Ignoring tflite-runtime: markers 'platform_system == "Linux"' don't match your environment
  Cloning https://github.com/tensorflow/docs to /private/var/folders/sq/kkc83jmn3_ddzwpqqq9wm6m40000gn/T/pip-req-build-cpacsot2
  Running command git clone --filter=blob:none --quiet https://github.com/tensorflow/docs /private/var/folders/sq/kkc83jmn3_ddzwpqqq9wm6m40000gn/T/pip-req-build-cpacsot2
  Resolved https://github.com/tensorflow/docs to commit 60c51e7a4b34ff7e02b339b361a5bff7f091cbf8
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Using cached keras-3.12.1-py3-none-any.whl.metadata (5.9 kB)
  Using cached tensorflow-2.20.0-cp310-cp310-macosx_12_0_arm64.whl.metadata (4.5 kB)
  Using cached tf_models_official-2.20.0-py2.py3-none-any.whl.metadata (1.6 kB)
  Using cached tqdm-4.67.3-py3-none-any.whl.metadata (57 kB)
  Using cached ipympl-0.10.0-py3-none-any.whl.metadata (9.4 kB)
  Using cached moviepy-2.2

In [4]:
%pip install opencv-python

Note: you may need to restart the kernel to use updated packages.


In [5]:
%pip install imageio

Note: you may need to restart the kernel to use updated packages.


In [6]:
%pip install tensorflow

  Using cached tensorflow-2.20.0-cp310-cp310-macosx_12_0_arm64.whl.metadata (4.5 kB)
  Using cached absl_py-2.4.0-py3-none-any.whl.metadata (3.3 kB)
  Using cached astunparse-1.6.3-py2.py3-none-any.whl.metadata (4.4 kB)
  Using cached flatbuffers-25.12.19-py2.py3-none-any.whl.metadata (1.0 kB)
  Using cached gast-0.7.0-py3-none-any.whl.metadata (1.5 kB)
  Using cached google_pasta-0.2.0-py3-none-any.whl.metadata (814 bytes)
  Using cached libclang-18.1.1-1-py2.py3-none-macosx_11_0_arm64.whl.metadata (5.2 kB)
  Using cached opt_einsum-3.4.0-py3-none-any.whl.metadata (6.3 kB)
  Using cached protobuf-6.33.5-cp39-abi3-macosx_10_9_universal2.whl.metadata (593 bytes)
  Using cached requests-2.32.5-py3-none-any.whl.metadata (4.9 kB)
  Using cached termcolor-3.3.0-py3-none-any.whl.metadata (6.5 kB)
  Using cached wrapt-2.1.1-cp310-cp310-macosx_11_0_arm64.whl.metadata (7.4 kB)
  Using cached grpcio-1.78.1-cp310-cp310-macosx_11_0_universal2.whl.metadata (3.8 kB)
  Using cached tensorboard-2.20.0

In [12]:
%pip install ai_edge_litert

  Using cached ai_edge_litert-2.1.2-cp310-cp310-macosx_12_0_arm64.whl.metadata (2.1 kB)
  Using cached backports_strenum-1.3.1-py3-none-any.whl.metadata (3.7 kB)
  Using cached tqdm-4.67.3-py3-none-any.whl.metadata (57 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.0/13.0 MB 16.0 MB/s  0:00:00m0:00:010:01
Using cached tqdm-4.67.3-py3-none-any.whl (78 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [ai_edge_litert]m [ai_edge_litert]
Note: you may need to restart the kernel to use updated packages.


In [13]:
import random
import os
import cv2
import numpy as np

# Some modules to display an animation using imageio.
import imageio

import tensorflow as tf
from ai_edge_litert.interpreter import Interpreter


## Building the model

In [14]:
print('Working directory:', os.getcwd())

Working directory: /Users/kausar/Documents/moment-to-action


In [15]:

# Create the interpreter and signature runner
tflite_filename = os.path.join(os.getcwd(), 'movinet/violence-streaming/model.tflite')

interpreter = Interpreter(model_path=tflite_filename)
runner = interpreter.get_signature_runner()

init_states = {
    name: tf.zeros(x['shape'], dtype=x['dtype'])
    for name, x in runner.get_input_details().items()
}
del init_states['image']


### Inference on Streaming

In [16]:
def to_gif(images,path = './animation.gif' ):
  converted_images = np.clip(images * 244, 0, 244).astype(np.uint8)
  imageio.mimsave(path, converted_images, fps=5)
  return embed.embed_file(path)

def frames_from_video_file(video_path, n_frames, output_size = (224,224), frame_step = 15):
  """
    Creates frames from each video file present for each category.

    Args:
      video_path: File path to the video.
      n_frames: Number of frames to be created per video file.
      output_size: Pixel size of the output frame image.

    Return:
      An NumPy array of frames in the shape of (n_frames, height, width, channels).
  """
  # Read each video frame by frame
  result = []
  src = cv2.VideoCapture(str(video_path))

  video_length = src.get(cv2.CAP_PROP_FRAME_COUNT)

  need_length = 1 + (n_frames - 1) * frame_step

  if need_length > video_length:
    start = 0
  else:
    max_start = video_length - need_length
    start = random.randint(0, max_start + 1)

  src.set(cv2.CAP_PROP_POS_FRAMES, start)
  # ret is a boolean indicating whether read was successful, frame is the image itself
  ret, frame = src.read()
  result.append(format_frames(frame, output_size))

  for _ in range(n_frames - 1):
    for _ in range(frame_step):
      ret, frame = src.read()
    if ret:
      frame = format_frames(frame, output_size)
      result.append(frame)
    else:
      result.append(np.zeros_like(result[0]))
  src.release()
  result = np.array(result)[..., [2, 1, 0]]

  return result

def format_frames(frame, output_size):
  """
    Pad and resize an image from a video.

    Args:
      frame: Image that needs to resized and padded.
      output_size: Pixel size of the output frame image.

    Return:
      Formatted frame with padding of specified output size.
  """
  frame = tf.image.convert_image_dtype(frame, tf.float32)
  frame = tf.image.resize_with_pad(frame, *output_size)
  return frame

def video_to_gif_tensor(video_path, image_size=(224, 224), fps=12):
    """
    Processes frames from a video file, saves them as a GIF in the same directory, and loads the GIF as a TensorFlow tensor.

    Args:
      video_path: String path to the input video file.
      image_size: Tuple indicating the size to which each frame should be resized.
      fps: Frames per second to be used in the GIF.

    Returns:
      A TensorFlow tensor representing the loaded GIF.
    """
    # Generate the gif_path in the same directory with a .gif extension
    gif_path = os.path.splitext(video_path)[0] + '.gif'

    # Assume frames_from_video_file is a function that extracts frames from video
    images = frames_from_video_file(video_path, n_frames=fps)  # function to be defined or replaced

    # Convert images to uint8 and save as GIF
    converted_images = np.clip(images * 255, 0, 255).astype(np.uint8)  # Proper scaling to 255
    imageio.mimsave(gif_path, converted_images, fps=fps)

    # Load the GIF file into a TensorFlow tensor
    raw = tf.io.read_file(gif_path)
    video = tf.io.decode_gif(raw)
    video = tf.image.resize(video, image_size)
    video = tf.cast(video, tf.float32) / 255.0  # Normalize to [0,1]

    return video

CLASSES = ['Fight','No_Fight']

def get_top_k(probs, k=2, label_map=CLASSES):
  """Outputs the top k model labels and probabilities on the given video."""
  top_predictions = tf.argsort(probs, axis=-1, direction='DESCENDING')[:k]
  top_labels = tf.gather(label_map, top_predictions, axis=-1)
  top_labels = [label.decode('utf8') for label in top_labels.numpy()]
  top_probs = tf.gather(probs, top_predictions, axis=-1).numpy()
  return tuple(zip(top_labels, top_probs))


In [19]:
# Insert your video clip here

clip_path = os.path.join(os.getcwd(), 'movinet/violence-streaming/test_videos/tf_X7GtOGyE_0.avi')

video = video_to_gif_tensor(clip_path, image_size=(172, 172))
clips = tf.split(video[tf.newaxis], video.shape[0], axis=1)

# To run on a video, pass in one frame at a time
states = init_states
for clip in clips:
  # Input shape: [1, 1, 172, 172, 3]
  outputs = runner(**states, image=clip)
  logits = outputs.pop('logits')[0]
  states = outputs

probs = tf.nn.softmax(logits)
top_k = get_top_k(probs)
print()
for label, prob in top_k:
  print(label, prob)




Fight 0.59705454
No_Fight 0.4029455


INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
